In [1]:
import numpy as np
import numpy.linalg as lin

np.set_printoptions(precision=3, linewidth=150, suppress=True)
np.random.seed(123)

## La simulation numérique

Si le prix des bananes est important, le secteur qui a les plus gros besoins en résolution de systèmes
matriciels est la simulation numérique.
Cela comprend <a href="https://www.nas.nasa.gov/SC18/demos/demo10.html">ceci</a>

<center><img alt="simulation du X-57 par la Nasa" src="images/nasa-x57-pression.jpg"/></center>

et tout ce qu'on simule numériquement à savoir tout ce qui touche le transport, l'énergie, la construction, tout
ce qu'on fabrique, qui coûte cher et qui doit avoir un comportement physique bien précis. Mais ce n'est pas tout,
comprendre notre environnement (météo, réchauffement de la planète, chimie...) revient aussi à résoudre des
systèmes matriciels. Cela étant, il existe aussi plein de méthodes de simulation numérique
qui ne passent pas par des systèmes matriciels.

Pour faire l'image ci-dessus, on transforme des équations de physique comme [Navier-Stokes](https://fr.wikipedia.org/wiki/%C3%89quations_de_Navier-Stokes) en un système matriciel où les inconnues sont définies en chaque point d'un maillage à définir. Dans notre cas l'inconnue est la pression et le maillage est l'intérieur d'une
boîte imaginaire qui comprend l'avion et l'air qui circule autour.

Si la boîte est un cube avec 1000 points dans chaque direction, on a 1 milliard de points dans la boite (moins ce qui est à l'intérieur de l'avion mais restons sur 1 milliards). Alors la matrice a 1 trillion d'éléments (un milliard au carré).

$$
\begin{bmatrix}
a_{11} \; a_{12} \ldots a_{1,10^9} \\
a_{21} \; a_{22} \ldots a_{2,10^9} \\
 \vdots \\
a_{10^9,1} a_{n2} \ldots a_{10^9,10^9} \\
\end{bmatrix}
\;
\begin{bmatrix}
p_{1} \\
p_{2} \\
\vdots \\
p_{10^9} \\
\end{bmatrix}
=
\begin{bmatrix}
f_{1} \\
f_{2} \\
\vdots \\
f_{10^9} \\
\end{bmatrix}
$$

Inverser une matrice se fait en $O(n^3)$ opérations soit $O(10^{27})$ dans notre cas. 

Si vous avez [l'ordinateur le plus puissant du monde](https://www.top500.org/)
vous pouvez faire 1 Eflops soit $10^{18}$ opérations par seconde en pic (en 2024). Aussi il vous faudra $10^{9}$ secondes soit un peu plus de 30 ans. C'est trop.

**Inverser une matrice ou résoudre par une méthode directe n'est pas la bonne solution pour résoudre un grand système matriciel.**

#### Exercice
Pour une telle simulation il faut aussi calculer la _vitesse_ de l'air en chacun des 1 milliard de points de notre maillage. Une vitesse c'est 3 variables qu'il faut ajouter à la pression. Combien de temps faut-il pour inverser la matrice qui intègre aussi la vitesse de l'air ?

In [2]:
# ceci est une calculatrice, vous pouvez donc écrire la réponse ici


Estimer le temps d'un gros calcul avant de le lancer est une bonne idée. 

D'ailleur avec des temps si longs, il ne
faut pas rester aux ordres de grandeurs mais préciser avec la formule exacte. Ainsi avec Choleski c'est $n^3/3$ opérations, donc 3 fois moins, mais bon...

## Méthodes itératives

Les méthodes itératives sont des méthodes qui s'approchent pas à pas de la solution recherchée. Elles permettent de trouver une approximation de ${\bf x}$, la solution de $A\, {\bf x} = b$.

On arrête le calcul lorsqu'on estime qu'on est à une distance choisie de la solution (qu'on appelle l'_erreur_) plutôt que d'attendre d'être à la solution exacte.
(De toute facon la notion de solution exacte sur un ordinateur qui est limité en précision est doutable. _On cherchera donc jamais à avoir une erreur plus petite que notre précision maximale_.)

L'idée fondamentale de l'algorithme itératif est d'avoir une formule du type $\; {\bf x}^{t+1} = N \, {\bf x}^t + {\bf b}\;$ ou en Python:

```
x = np.random(size = b.size)
while np.square(x - old_x) > seuil:
    old_x = x
    x = N @ x + b
```

La magie c'est si **x** converge. Dans ce cas on a atteint un point fixe,
c. à d. que ${\bf x}^{t+1} = {\bf x}^t$ et donc 

$${\bf x}^t = N\, {\bf x}^t + {\bf b} \quad\text{c. à d.}\quad (I-N)\, {\bf x}^t = {\bf b}$$

On retrouve notre $A \; {\bf x} = {\bf b}$ en posant $N=I-A$.

## Méthode de Jacobi

La méthode de Jacobi n'utilise pas $N=I-A$ car en général cela ne converge pas.
À la place on découpe la matrice $A$ en $M$ et $N$ avec

* $M$ la matrice diagonale qui comprend les éléments de la diagonale de $A$
* $N = M - A$  (donc 0 sur la diagonale et l'opposé des éléments de $A$ ailleurs)

Ainsi le système à résoudre est $\; (M - N) \, {\bf x} = {\bf b}$.

La formule itérative est donc pour l'itération k+1 exprimée en fonction de l'itération k :

$$
M \; {\bf x}^{k+1} =  N\; {\bf x}^k + {\bf b}
$$

et comme $M$ est une matrice diagonale :

$$
\begin{array}{l}
a_{11} x_1^{k+1} = \qquad\quad -a_{12} \, x_2^k - a_{13} \, x_3^k  \ldots - a_{1n} \, x_n^k + b_1 \\
a_{22} x_2^{k+1} = -a_{21} \, x_1^k \qquad\quad - a_{23} \, x_3^k  \ldots - a_{2n} \, x_n^k + b_2 \\
a_{33} x_3^{k+1} = -a_{31} \, x_1^k - a_{32} \, x_2^k  \qquad\quad \ldots - a_{3n} \, x_n^k + b_3 \\
 \vdots \\
a_{nn} x_n^{k+1} = -a_{n1} \, x_1^k - a_{n2} \, x_2^k  \quad \ldots - a_{n-1,n-1} \, x_{n-1}^k + b_n \\
\end{array}
$$

On voit en filigramme $A \; {\bf x} = {\bf b}$.

Pour calculer $x_1^{k+1}$ il faut diviser le terme de droite de la première ligne par $a_{11}$ il faut donc que  $a_{11} \ne 0$.
Pour calculer les termes suivants $x_i^{k+1}$ il faut aussi que $a_{ii}$ soient non nul donc
_il faut que $A$ n'ait pas pas de zéro sur sa diagonale_.

Cela se retrouve dans l'écriture suivante qui reprend la formule initiale :
$ {\bf x}^{k+1} =  M^{-1} \;(N\; {\bf x}^k + {\bf b})$ 
On voit que pour être efficace, il faut que $M$ soit facilement inversible, sinon autant inverser $A$ directement.
Ici c'est bien le cas puisque $M$ est diagonale.

#### Programmons Jacobi

In [5]:
A = np.random.randint(10, size=(4,4))
b = A.sum(axis=1)                     # ainsi la solution est [1,1,1,1]
print('A:\n', A, "\nb:\n", b, "\n")

M = np.diag(A)        # attention, c'est un vecteur
N = np.diag(M) - A    # np.diag d'une matrice donne un vecteur, np.diag d'un vecteur donne une matrice
print(f"M:\n {np.diag(M)}\nN:\n {N}\n")

x = np.random.random(4)
for i in range(20):
    print(f"x_{i} = {x}")
    x = (N @ x + b) / M

A:
 [[3 5 9 0]
 [8 1 6 3]
 [3 5 9 7]
 [9 2 3 3]] 
b:
 [17 18 24 17] 

M:
 [[3 0 0 0]
 [0 1 0 0]
 [0 0 9 0]
 [0 0 0 3]]
N:
 [[ 0 -5 -9  0]
 [-8  0 -6 -3]
 [-3 -5  0 -7]
 [-9 -2 -3  0]]

x_0 = [0.271 0.53  0.175 0.315]
x_1 = [ 4.257 13.839  2.037  4.326]
x_2 = [-23.509 -41.258  -9.806 -18.368]
x_3 = [103.847 320.008  47.71  113.505]
x_4 = [ -670.811 -1439.552  -298.013  -566.923]
x_5 = [3298.958 8873.331 1466.961 3275.812]
x_6 = [-19184.103 -45002.866  -8574.468 -17273.722]
x_7 = [100733.848 256758.796  44834.077  96134.354]
x_8 = [ -562427.891 -1363260.308  -250990.222  -518202.486]
x_9 = [3025076.847 7559989.919 1347891.846 2847119.767]
x_10 = [-16643653.068 -40829307.148  -7422777.167 -15463109.998]
x_11 = [9.032e+07 2.241e+08 4.026e+07 8.457e+07]
x_12 = [-4.942e+08 -1.218e+09 -2.204e+08 -4.606e+08]
x_13 = [2.691e+09 6.658e+09 1.200e+09 2.515e+09]
x_14 = [-1.470e+10 -3.627e+10 -6.552e+09 -1.371e+10]
x_15 = [8.010e+10 1.980e+11 3.571e+10 7.482e+10]
x_16 = [-4.371e+11 -1.080e+12 -1.949e

Cela ne converge pas vraiment... (si vous relancez et voyez des `NaN` c'est qu'il y a un zéro sur la diagonale)

2e essai :

In [9]:
A = np.random.randint(10, size=(4,4))
A = A + np.diag(A.sum(axis=0))  # voir ci-dessous
b = A.sum(axis=1)                     # ainsi la solution est [1,1,1,1]
print('A:\n', A, "\nb:\n", b, "\n")


M = np.diag(A)        # attention, c'est un vecteur
N = np.diag(M) - A    # np.diag d'une matrice donne un vecteur, np.diag d'un vecteur donne une matrice
print(f"M:\n {np.diag(M)}\nN:\n {N}\n")

x = np.random.random(4)
for i in range(20):
    print(f"x_{i} = {x}")
    x = (N @ x + b) / M

A:
 [[14  2  9  3]
 [ 2 26  7  9]
 [ 1  3 41  3]
 [ 7  9  9 21]] 
b:
 [28 44 48 46] 

M:
 [[14  0  0  0]
 [ 0 26  0  0]
 [ 0  0 41  0]
 [ 0  0  0 21]]
N:
 [[ 0 -2 -9 -3]
 [-2  0 -7 -9]
 [-1 -3  0 -3]
 [-7 -9 -9  0]]

x_0 = [0.065 0.005 0.887 0.911]
x_1 = [1.234 1.133 1.102 1.786]
x_2 = [0.747 0.682 0.927 0.821]
x_3 = [1.131 1.101 1.042 1.252]
x_4 = [0.904 0.891 0.971 0.895]
x_5 = [1.057 1.052 1.018 1.091]
x_6 = [0.962 0.959 0.988 0.951]
x_7 = [1.024 1.023 1.007 1.035]
x_8 = [0.984 0.984 0.995 0.979]
x_9 = [1.01  1.01  1.003 1.014]
x_10 = [0.994 0.994 0.998 0.991]
x_11 = [1.004 1.004 1.001 1.006]
x_12 = [0.997 0.997 0.999 0.996]
x_13 = [1.002 1.002 1.001 1.002]
x_14 = [0.999 0.999 1.    0.998]
x_15 = [1.001 1.001 1.    1.001]
x_16 = [1.    1.    1.    0.999]
x_17 = [1. 1. 1. 1.]
x_18 = [1. 1. 1. 1.]
x_19 = [1. 1. 1. 1.]


Magie !

### Pourquoi le 2e cas marche ?

Pour qu'une méthode itérative du type  ${\bf x} = B\; {\bf x} + {\bf c}$  converge il faut au choix

* $\rho(B) < 1\quad$ avec $\rho$ le rayon spectral (qui est la plus grande valeur propre en valeur absolue)
* $||B|| < 1\quad$ avec une norme matricielle qui est subordonnée à une norme vectorielle.
             

#### Cas de la méthode de Jacobi

Pour la matrice $B$ de Jacobi, on respecte ces conditions si $A$ est à **diagonale dominante** à savoir que chaque élément de la diagonale est plus grand en module que la sommes de tous les autres éléments en module de sa ligne ou colonne : $|a_{i,i}| \ge \sum_{j \ne i} |a_{i,j}|$.

Jacobi converge aussi si A est symétrique, réelle et définie positive 
(c. à d. $\forall {\bf x}, \; {\bf x}^T \, A \, {\bf x} > 0$).

### Temps de calcul

Si on converge en 10 itérations alors
on utilise 10 multiplications matricielles, 10 additions vectorielles et 10 divisions vectorielles soit 180 opérations,
à comparer aux $4^3 / 3 = 22$ opérations d'une méthode directe. Oups !

Quelques raisons qui font qu'on y perd :

* La matrice $A$ est trop petite, les méthodes itératives sont intéressantes pour les grandes matrice
* La méthode de Jacobi n'est pas la meilleure (mais elle est très facilement parallélisable)
* Le rayon spectral de la matrice est trop grand. Plus le rayon spectral est
  petit et plus on converge vite.